In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#load data set
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("radheshyamkollipara/bank-customer-churn")

print("Path to dataset files:", path)

# 3. Load the CSV file into a pandas DataFrame
# Replace 'Customer-Churn-Records.csv' if the filename listed above is different
csv_path = os.path.join(path, "Customer-Churn-Records.csv")
df = pd.read_csv(csv_path)

# 4. Inspect the DataFrame
#print(df.head())

Using Colab cache for faster access to the 'bank-customer-churn' dataset.
Path to dataset files: /kaggle/input/bank-customer-churn


In [ ]:
df.columns
# Drop unnecessary identification columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [ ]:
# Check the first few rows and column types
display(df.head())
print(df.info())

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CreditScore         10000 non-null  int64  
 1   Geography           10000 non-null  object 
 2   Gender              10000 non-null  object 
 3   Age                 10000 non-null  int64  
 4   Tenure              10000 non-null  int64  
 5   Balance             10000 non-null  float64
 6   NumOfProducts       10000 non-null  int64  
 7   HasCrCard           10000 non-null  int64  
 8   IsActiveMember      10000 non-null  int64  
 9   EstimatedSalary     10000 non-null  float64
 10  Exited              10000 non-null  int64  
 11  Complain            10000 non-null  int64  
 12  Satisfaction Score  10000 non-null  int64  
 13  Card Type           10000 non-null  object 
 14  Point Earned        10000 non-null  int64  
dtypes: float64(2), int64(10), object(3)
memory usage: 1.1+

In [ ]:
# Convert categorical text columns (Geography, Gender) into numerical dummy columns
df = pd.get_dummies(df, columns=['Geography', 'Gender', "Card Type"], drop_first=True)

In [ ]:
display(df.head())

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Point Earned,Geography_Germany,Geography_Spain,Gender_Male,Card Type_GOLD,Card Type_PLATINUM,Card Type_SILVER
0,619,42,2,0.00,1,1,1,101348.88,1,1,2,464,False,False,False,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,1,3,456,False,True,False,False,False,False
2,502,42,8,159660.80,3,1,0,113931.57,1,1,3,377,False,False,False,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,0,5,350,False,False,False,True,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,0,5,425,False,True,False,True,False,False


In [ ]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) from the target label (y)
X = df.drop('Exited', axis=1)
y = df['Exited']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to a DataFrame for clean tracking
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

display(X_scaled.head())


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Complain,Satisfaction Score,Point Earned,Geography_Germany,Geography_Spain,Gender_Male,Card Type_GOLD,Card Type_PLATINUM,Card Type_SILVER
0,-0.326221,0.293517,-1.041760,-1.225848,-0.911583,0.646092,0.970243,0.021886,1.972908,-0.721130,-0.630839,-0.578736,-0.573809,-1.095988,-0.577658,-0.57658,-0.576734
1,-0.440036,0.198164,-1.387538,0.117350,-0.911583,-1.547768,0.970243,0.216534,1.972908,-0.009816,-0.666251,-0.578736,1.742740,-1.095988,-0.577658,-0.57658,-0.576734
2,-1.536794,0.293517,1.032908,1.333053,2.527057,0.646092,-1.030670,0.240687,1.972908,-0.009816,-1.015942,-0.578736,-0.573809,-1.095988,-0.577658,-0.57658,-0.576734
3,0.501521,0.007457,-1.387538,-1.225848,0.807737,-1.547768,-1.030670,-0.108918,-0.506866,1.412812,-1.135457,-0.578736,-0.573809,-1.095988,1.731128,-0.57658,-0.576734
4,2.063884,0.388871,-1.041760,0.785728,-0.911583,0.646092,0.970243,-0.365276,-0.506866,1.412812,-0.803472,-0.578736,1.742740,-1.095988,1.731128,-0.57658,-0.576734


In [ ]:
#split dataset
from sklearn.model_selection import train_test_split

# Split data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")


Training set shape: (8000, 17)
Testing set shape: (2000, 17)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest model with class weighting
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# Train the model on the training data
rf_model.fit(X_train, y_train)

print("Model training complete!")


Model training complete!


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Make predictions on the test set
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1] # Probability of churning

# Print evaluation metrics
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")


--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1592
           1       1.00      1.00      1.00       408

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

ROC-AUC Score: 0.9993


In [ ]:
import joblib

# Save the model and scaler to disk
joblib.dump(rf_model, 'churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Model and scaler successfully saved as 'churn_model.pkl' and 'scaler.pkl'!")


Model and scaler successfully saved as 'churn_model.pkl' and 'scaler.pkl'!


In [ ]:
import pandas as pd
import numpy as np
import joblib

# Load the saved model and scaler
loaded_model = joblib.load('churn_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

print("Model and scaler loaded successfully!")


Model and scaler loaded successfully!


In [ ]:
def churn_agent_decision(customer_dict):
    """
    An agentic function that takes customer data, uses the ML model tool
    to predict churn risk, and determines a retention action.
    """
    # 1. Convert input dictionary to DataFrame
    input_df = pd.DataFrame([customer_dict])

    # Ensure columns match the training features structure (handling dummy columns)
    # For simplicity in this demo, let's assume the input dictionary matches your raw feature set
    # and we map categorical values correctly.

    print(f"--- Analyzing Customer: {customer_dict.get('Surname', 'Valued Customer')} ---")
    print(f"Credit Score: {customer_dict['CreditScore']} | Balance: ${customer_dict['Balance']:,.2f} | Age: {customer_dict['Age']}")

    # Mocking the preprocessing pipeline step for a single row prediction
    # (In a full app, you'd pass this through the exact same pandas/scaler pipeline)

    # For demonstration, let's simulate passing a sample row through the model
    # Let's pull a random row from our original dataset to test the agent live:
    return "Agent initialized and ready for execution."

print(churn_agent_decision({'CreditScore': 650, 'Age': 45, 'Balance': 125000.0, 'Surname': 'Smith'}))


--- Analyzing Customer: Smith ---
Credit Score: 650 | Balance: $125,000.00 | Age: 45
Agent initialized and ready for execution.


In [ ]:
# Grab a sample customer from your test data
sample_customer_scaled = X_test.iloc[[0]]
actual_label = y_test.iloc[0]

# Agent makes the prediction
churn_prob = loaded_model.predict_proba(sample_customer_scaled)[0][1]
prediction = loaded_model.predict(sample_customer_scaled)[0]

print(f"\n[Agent Tool Output]")
print(f"Calculated Churn Probability: {churn_prob * 100:.2f}%")

# Agent Reasoning and Action
if churn_prob > 0.5:
    print("🤖 Agent Decision: HIGH RISK DETECTED.")
    print("✉️ Action Taken: Automatically generated retention email sent with a 20% fee waiver and loyalty bonus!")
else:
    print("🤖 Agent Decision: Low risk. Customer is stable.")
    print("✉️ Action Taken: Scheduled for standard monthly newsletter.")



[Agent Tool Output]
Calculated Churn Probability: 0.00%
🤖 Agent Decision: Low risk. Customer is stable.
✉️ Action Taken: Scheduled for standard monthly newsletter.


In [ ]:
# 1. Install Gradio inside Colab (if not already installed)
!pip install -q gradio

import gradio as gr
import pandas as pd
import joblib

# 2. Load the trained model and scaler
model = joblib.load('churn_model.pkl')
scaler = joblib.load('scaler.pkl')

# 3. Define prediction function matching ALL dataset features
def predict_churn(credit_score, age, tenure, balance, num_products, has_card, is_active,
                  salary, geography, gender, card_type, complain, satisfaction_score, points):

    # Map raw inputs to match encoded training schema
    raw_data = {
        'CreditScore': credit_score,
        'Age': age,
        'Tenure': tenure,
        'Balance': balance,
        'NumOfProducts': num_products,
        'HasCrCard': 1 if has_card else 0,
        'IsActiveMember': 1 if is_active else 0,
        'EstimatedSalary': salary,
        'Complain': 1 if complain else 0,
        'Satisfaction Score': satisfaction_score,
        'Point Earned': points,
        'Geography_Germany': 1 if geography == 'Germany' else 0,
        'Geography_Spain': 1 if geography == 'Spain' else 0,
        'Gender_Male': 1 if gender == 'Male' else 0,
        'Card Type_GOLD': 1 if card_type == 'GOLD' else 0,
        'Card Type_PLATINUM': 1 if card_type == 'PLATINUM' else 0,
        'Card Type_SILVER': 1 if card_type == 'SILVER' else 0,
    }

    input_df = pd.DataFrame([raw_data])

    # Reorder columns dynamically to match the fitted scaler exact order
    if hasattr(scaler, 'feature_names_in_'):
        input_df = input_df[list(scaler.feature_names_in_)]

    # Scale inputs using saved scaler
    scaled_input = scaler.transform(input_df)

    # Predict churn probability
    churn_prob = model.predict_proba(scaled_input)[0][1]
    churn_pct = churn_prob * 100

    # Agent Logic & Action Formatting
    if churn_prob > 0.5:
        status = f"🔴 HIGH RISK ({churn_pct:.1f}% Churn Probability)"
        action = (
            "🤖 Agent Action Triggered:\n"
            "• Drafted personalized retention offer email\n"
            "• Applied 20% fee waiver for 6 months\n"
            "• Escalated profile to Senior Relationship Manager"
        )
    else:
        status = f"🟢 LOW RISK ({churn_pct:.1f}% Churn Probability)"
        action = (
            "🤖 Agent Action Triggered:\n"
            "• Customer account stable\n"
            "• Enrolled in standard monthly updates queue"
        )

    return status, action

# 4. Build complete Gradio Interface
demo = gr.Interface(
    fn=predict_churn,
    inputs=[
        gr.Slider(300, 850, value=650, label="Credit Score"),
        gr.Slider(18, 100, value=40, label="Age"),
        gr.Slider(0, 10, value=3, label="Tenure (Years)"),
        gr.Number(value=50000.0, label="Account Balance ($)"),
        gr.Slider(1, 4, value=1, step=1, label="Number of Products"),
        gr.Checkbox(value=True, label="Has Credit Card?"),
        gr.Checkbox(value=True, label="Is Active Member?"),
        gr.Number(value=75000.0, label="Estimated Salary ($)"),
        gr.Dropdown(["France", "Germany", "Spain"], value="France", label="Geography"),
        gr.Radio(["Female", "Male"], value="Female", label="Gender"),
        gr.Dropdown(["DIAMOND", "GOLD", "SILVER", "BRONZE"], value="GOLD", label="Card Type"),
        gr.Checkbox(value=False, label="Has Registered Complain?"),
        gr.Slider(1, 5, value=3, step=1, label="Satisfaction Score (1-5)"),
        gr.Number(value=500, label="Point Earned"),
    ],
    outputs=[
        gr.Textbox(label="Risk Assessment"),
        gr.Textbox(label="Agent Decision & Action"),
    ],
    title="🏦 Bank Customer Churn Agent",
    description="Interactive UI matching Kaggle Bank Customer Churn features.",
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bbba40a15240a514be.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
